# MHEAT Datalab Quickstart — using the `mheat-client` Python SDK

This notebook is the 5-minute on-ramp for **EDITO Datalab** users — marine biologists, climate scientists, ML engineers, and API integrators — who want to pull Mediterranean marine-heatwave detections + the underlying ARCO Zarr SST cube directly into a JupyterLab session.

We use the thin Python SDK [`mheat-client`](../sdk/README.md), which wraps the MHEAT REST API in a single `MheatClient` class that returns native scientific-Python types (`dict`, `pandas.DataFrame`, `xarray.Dataset`).

**What you'll do in five cells:**

1. Install the SDK + import it
2. Connect to the live backend; print health, freshness, extent
3. Pull the 2022 Mediterranean summer events as a DataFrame
4. Open the ARCO Zarr cube and plot the SST anomaly for a single day
5. Pick the hottest event and plot its SST/climatology/threshold time series

**Prerequisites:** A running MHEAT backend reachable at `http://localhost:8000` (or set `MHEAT_BASE_URL` below to your deployed instance — e.g. on the EDITO Onyxia platform).

---

## Step 1 — Install the SDK

From the MHEAT monorepo, install the SDK in editable mode. On a Datalab kernel without the repo cloned, swap this for `pip install mheat-client` once the package is on PyPI (see `sdk/README.md` § *Publishing*).

We also pull the `[plot]` extra so we can use matplotlib in cell 4 + 5.

In [ ]:
# From the repo root: pip install -e ./sdk[plot]
# Inside this notebook (which lives at tutorials/), the SDK is one level up:
%pip install -q -e "../sdk[plot]"

import os
import matplotlib.pyplot as plt
import pandas as pd
from mheat_client import MheatClient, __version__

print(f"mheat-client v{__version__} ready")

## Step 2 — Connect to the backend

The three smallest endpoints (`health`, `freshness`, `extent`) tell us:
- **health** — is the service alive? (and what version)
- **freshness** — when was the SST cube last updated? Is a pull in progress?
- **extent** — what date range does the anomaly cube cover? What is the colour-scale range?

These are the three things you should always check at the top of a notebook so you know what you're working with.

In [ ]:
BASE_URL = os.environ.get("MHEAT_BASE_URL", "http://localhost:8000")
client = MheatClient(BASE_URL, timeout=180.0)

print("--- /api/health ---")
print(client.health())

print("\n--- /api/freshness ---")
fresh = client.freshness()
for k, v in fresh.items():
    print(f"  {k}: {v}")

print("\n--- /api/anomaly/extent ---")
ext = client.extent()
for k, v in ext.items():
    print(f"  {k}: {v}")

## Step 3 — Pull events as a DataFrame

The classic MHEAT workflow: ask for clustered MHW events in a date window + minimum Hobday category, get back a tidy `pandas.DataFrame` with one row per event.

Under the hood, the SDK fetches `/api/events.parquet` (GeoParquet) — much smaller and faster to parse than the equivalent GeoJSON for archives that span 30 years. The `geometry` column is **WKB bytes** (well-known binary); pass `as_geodataframe=True` if you have `geopandas` installed and want a `GeoDataFrame` with shapely polygons.

We sort by **peak intensity** to find the most extreme events of the 2022 Mediterranean summer.

In [ ]:
events = client.events(
    start="2022-05-15",
    end="2022-09-15",
    min_category=1,    # keep all categories so we see the full distribution
)
print(f"{len(events)} events in 2022-05-15 .. 2022-09-15\n")

# Top 5 by peak intensity (excluding the heavy WKB geometry column for display)
show_cols = [
    "event_id", "category_name", "date_start", "date_end", "date_peak",
    "intensity_max", "intensity_mean", "intensity_cumulative", "duration_days",
    "n_aquaculture_sites", "mpa_area_km2", "centroid_lon", "centroid_lat",
]
top5 = events.sort_values("intensity_max", ascending=False).head(5)[show_cols]
top5

## Step 4 — Open the ARCO SST cube + plot one day

`client.sst_cube()` opens the published ARCO Zarr store at `/api/data/sst.zarr` via `xarray.open_zarr(consolidated=True)`. Nothing is downloaded yet — xarray reads only the consolidated metadata, so the entire 30-year cube appears as a lazy dataset that you can slice/aggregate before any bytes flow.

Below we load one day's slice (the peak of the 2022 summer event) and plot it. The cube already contains the **anomaly-friendly** climatology + threshold so a reader can see what "normal" was and how far above it we are.

In [ ]:
cube = client.sst_cube()
print(cube)

# Pick the variable name (CMS often calls it analysed_sst).
sst_var = next((v for v in cube.data_vars if "sst" in v.lower() or "temp" in v.lower()),
               list(cube.data_vars)[0])
print(f"\nUsing variable: {sst_var}")

# Slice one day at the heart of the 2022 Med MHW.
DAY = "2022-08-15"
day_slice = cube[sst_var].sel(time=DAY, method="nearest")

# Plot — matplotlib via xarray. (Cartopy not required for this quickstart;
# the lon/lat axes already give you a usable map for the Mediterranean.)
fig, ax = plt.subplots(figsize=(11, 4.5))
day_slice.plot(ax=ax, cmap="RdYlBu_r", robust=True, cbar_kwargs={"label": f"{sst_var} (degC)"})
ax.set_title(f"Mediterranean SST — {DAY}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## Step 5 — Time series of the hottest event

Pick the row with the highest `intensity_max` from cell 3, then ask the API for the **per-pixel SST + Hobday climatology + 90th-percentile threshold** time series at that event's centroid. The SDK packages it as a date-indexed `DataFrame` with three columns; the underlying endpoint is `/api/events/{event_id}/series`.

We shade the area where SST exceeds the threshold — that's the MHW envelope as Hobday et al. (2016) defined it.

In [ ]:
hottest = events.sort_values("intensity_max", ascending=False).iloc[0]
print("Hottest event in window:")
print(f"  id           = {hottest['event_id']}")
print(f"  category     = {hottest['category_name']}")
print(f"  date_peak    = {hottest['date_peak'].date() if hasattr(hottest['date_peak'],'date') else hottest['date_peak']}")
print(f"  intensity_max= {hottest['intensity_max']:.2f} degC above threshold")
print(f"  centroid     = ({hottest['centroid_lon']:.3f}E, {hottest['centroid_lat']:.3f}N)")
print(f"  impact       = {int(hottest['n_aquaculture_sites'])} aquaculture sites, "
      f"{hottest['mpa_area_km2']:.0f} km^2 MPA")

ts = client.event_series(
    event_id=str(hottest["event_id"]),
    lon=float(hottest["centroid_lon"]),
    lat=float(hottest["centroid_lat"]),
    pad_days=14,
)
print(f"\n{len(ts)} daily samples around the event ({ts.index.min().date()} .. {ts.index.max().date()})")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts.index, ts["sst"],    label="SST",                    color="black", lw=1.0)
ax.plot(ts.index, ts["seas"],   label="Climatology (DOY mean)", color="tab:blue")
ax.plot(ts.index, ts["thresh"], label="90th-percentile threshold", color="tab:orange", ls="--")
ax.fill_between(
    ts.index, ts["thresh"], ts["sst"],
    where=(ts["sst"] > ts["thresh"]),
    interpolate=True, color="red", alpha=0.25, label="MHW envelope",
)
ax.set_title(f"{hottest['event_id']} -- {hottest['category_name']} "
             f"-- pixel ({ts.attrs['lat']:.2f}N, {ts.attrs['lon']:.2f}E)")
ax.set_ylabel("SST (degC)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Where to go next

- **Decode polygons** for spatial analysis: `client.events(..., as_geodataframe=True)` returns a `GeoDataFrame` (requires `pip install 'mheat-client[geo]'`). Then overlay on a basemap, intersect with your own field-survey points, etc.
- **Analyse trends across decades** — change `start='1996-01-01'` to pull the full 30-year archive in one call. The Parquet payload is small enough (~1 MB / 1000 events).
- **Train an ML model** — pull the cube via `client.sst_cube()` + the labels via `client.events()` and you have an SST-with-event-labels dataset ready for a temporal CNN. See `docs/ml_benchmark.md` (in progress) for the canonical train/val/test split.
- **Hit the OGC API endpoints** directly from QGIS — `http://localhost:8000/api/ogcapi/collections/mhw-events` works as a WFS source.
- **Read the SDK source** at `sdk/mheat_client/client.py` — it's ~150 lines and easy to extend.

## Cleanup

Close the HTTP connection pool when you're done.

In [ ]:
client.close()
print("Done.")